# Part 2 — Data Models & Prompt Templates

This notebook demonstrates the correct functioning of data modeling and prompt modules (EP-02, EP-06, EP-11, EP-15):

| File | Responsibility |
|------|----------------|
| `src/models/schemas.py` | All Pydantic v2 models of the pipeline |
| `src/models/state.py` | `BuilderState` and `QueryState` (`TypedDict`) for LangGraph |
| `src/prompts/templates.py` | Centralized catalog of prompt templates (PT-01 → PT-11) |
| `src/prompts/few_shot.py` | Loaders and formatters for few-shot examples |

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. Ingestion Models — `Document` and `Chunk`

`Document` represents a page extracted from a PDF. `Chunk` is a fragment produced by `RecursiveCharacterTextSplitter` with provenance metadata.

In [ ]:
from pydantic import ValidationError
from src.models.schemas import Document, Chunk

# Document: raw page from PDF
doc = Document(
    text="A Customer is any individual that has registered on the platform.",
    metadata={"source": "business_glossary.pdf", "page": "1"},
)
print("Document:")
print(f"  text (first 60): '{doc.text[:60]}'")
print(f"  metadata: {doc.metadata}")
print()

# Chunk: semantically coherent fragment
chunk = Chunk(
    text="A Customer is any individual that has registered on the platform.",
    chunk_index=0,
    metadata={"source": "business_glossary.pdf", "page": "1", "token_count": "14"},
)
print("Chunk:")
print(f"  chunk_index: {chunk.chunk_index}")
print(f"  token_count: {chunk.metadata['token_count']}")

# Pydantic validation — missing required field
try:
    bad = Chunk(text="test")  # missing chunk_index
except ValidationError as e:
    print(f"\n✅ ValidationError correctly detected: {e.error_count()} error(s)")

## 2. Extraction Models — `Triplet` and `TripletExtractionResponse`

`Triplet` captures a semantic fact (subject, predicate, object) with verbatim provenance and confidence. `TripletExtractionResponse` is the SLM's JSON output schema.

In [ ]:
from src.models.schemas import Triplet, TripletExtractionResponse

triplet = Triplet(
    subject="Customer",
    predicate="places",
    object="SalesOrder",
    provenance_text="A Customer places one or more SalesOrders.",
    confidence=1.0,
    source_chunk_index=0,
)
print("Triplet:")
print(f"  ({triplet.subject}, {triplet.predicate}, {triplet.object})")
print(f"  provenance: '{triplet.provenance_text}'")
print(f"  confidence: {triplet.confidence}")

# Composite SLM response
response = TripletExtractionResponse(
    triplets=[
        triplet,
        Triplet(
            subject="SalesOrder",
            predicate="contains",
            object="OrderLineItem",
            provenance_text="A SalesOrder contains one or more OrderLineItems.",
            confidence=1.0,
        ),
    ]
)
print(f"\nTripletExtractionResponse: {len(response.triplets)} triplet(s)")
for t in response.triplets:
    print(f"  ({t.subject}, {t.predicate}, {t.object}) [conf={t.confidence}]")

# Validate confidence range [0.0, 1.0]
try:
    bad = Triplet(
        subject="X", predicate="y", object="Z",
        provenance_text="...", confidence=1.5  # out of range
    )
except ValidationError as e:
    print(f"\n✅ Confidence > 1.0 detected: {e.errors()[0]['msg']}")

## 3. Entity Resolution — `EntityCluster`, `CanonicalEntityDecision`, `Entity`

In [ ]:
from src.models.schemas import EntityCluster, CanonicalEntityDecision, Entity

# Cluster of duplicate candidates found by K-NN blocking
cluster = EntityCluster(
    canonical_candidate="Customer",
    variants=["Customer", "customer", "Client", "client"],
    avg_similarity=0.91,
)
print("EntityCluster:")
print(f"  canonical_candidate: {cluster.canonical_candidate}")
print(f"  variants: {cluster.variants}")
print(f"  avg_similarity: {cluster.avg_similarity}")

# LLM Judge decision
decision = CanonicalEntityDecision(
    merge=True,
    canonical_name="Customer",
    reasoning="All variants refer to the same business entity concept: the platform customer.",
)
print(f"\nCanonicalEntityDecision: merge={decision.merge}, canonical='{decision.canonical_name}'")

# Resolved entity ready for RAG Mapping
entity = Entity(
    name="Customer",
    definition="Any individual or legal entity that has registered and made at least one purchase.",
    synonyms=["client", "buyer", "account holder"],
    provenance_text="A Customer is any individual or legal entity that has registered an account.",
    source_doc="business_glossary.pdf",
)
print(f"\nEntity: '{entity.name}'")
print(f"  definition: {entity.definition[:60]}...")
print(f"  synonyms: {entity.synonyms}")
print(f"  embedding: {'<empty — not yet computed>' if entity.embedding is None else len(entity.embedding)}")

## 4. DDL Schema — `ColumnSchema`, `TableSchema`, `EnrichedTableSchema`

In [ ]:
from src.models.schemas import ColumnSchema, TableSchema, EnrichedTableSchema, EnrichedColumn

# DDL table columns
columns = [
    ColumnSchema(name="CUST_ID", data_type="INT", is_primary_key=True),
    ColumnSchema(name="CUST_NM", data_type="VARCHAR(100)"),
    ColumnSchema(name="CUST_EMAIL", data_type="VARCHAR(255)"),
    ColumnSchema(name="REG_DT", data_type="DATE"),
]

table = TableSchema(
    table_name="TB_CUST",
    schema_name="dbo",
    columns=columns,
    ddl_source="CREATE TABLE dbo.TB_CUST (CUST_ID INT PRIMARY KEY, ...);",
)
print("TableSchema:")
print(f"  {table.schema_name}.{table.table_name} — {len(table.columns)} columns")
for col in table.columns:
    pk = " [PK]" if col.is_primary_key else ""
    print(f"  · {col.name}: {col.data_type}{pk}")

# Promotion to EnrichedTableSchema (adds human-readable names from LLM)
enriched = EnrichedTableSchema.from_table_schema(table)
enriched.enriched_table_name = "Customer Table"
enriched.table_description = "Stores master data for registered customers."
enriched.enriched_columns = [
    EnrichedColumn(original_name="CUST_ID", enriched_name="Customer ID"),
    EnrichedColumn(original_name="CUST_NM", enriched_name="Customer Name"),
    EnrichedColumn(original_name="CUST_EMAIL", enriched_name="Customer Email"),
    EnrichedColumn(original_name="REG_DT", enriched_name="Registration Date"),
]
print(f"\nEnrichedTableSchema:")
print(f"  enriched_table_name: '{enriched.enriched_table_name}'")
print(f"  description: '{enriched.table_description}'")
for ec in enriched.enriched_columns:
    print(f"  · {ec.original_name} → '{ec.enriched_name}'")

## 5. Mapping & Cypher — `MappingProposal`, `CriticDecision`, `CypherStatement`

In [ ]:
from src.models.schemas import MappingProposal, CriticDecision, CypherStatement, GraderDecision, RetrievedChunk

# LLM mapping proposal (RAG Mapper)
proposal = MappingProposal(
    table_name="TB_CUST",
    mapped_concept="Customer",
    confidence=0.95,
    reasoning="TB_CUST stores customer identity data matching the 'Customer' business concept.",
    alternative_concepts=["CustomerAddress"],
)
print("MappingProposal:")
print(f"  {proposal.table_name} → '{proposal.mapped_concept}' [conf={proposal.confidence}]")
print(f"  reasoning: {proposal.reasoning[:70]}...")

# LLM Critic decision
critic = CriticDecision(
    approved=True,
    critique=None,
)
print(f"\nCriticDecision: approved={critic.approved}")

# Validated Cypher statement
cypher_stmt = CypherStatement(
    cypher="MERGE (t:PhysicalTable {name: $table_name}) MERGE (c:BusinessConcept {name: $concept_name}) MERGE (t)-[:IMPLEMENTS]->(c)",
    params={"table_name": "TB_CUST", "concept_name": "Customer"},
    mapping_id="TB_CUST__Customer",
)
print(f"\nCypherStatement [id={cypher_stmt.mapping_id}]:")
print(f"  {cypher_stmt.cypher[:80]}...")
print(f"  params: {cypher_stmt.params}")

# GraderDecision (Self-RAG)
grade = GraderDecision(grounded=True, critique=None, action="pass")
print(f"\nGraderDecision: grounded={grade.grounded}, action='{grade.action}'")

## 6. LangGraph States — `BuilderState` and `QueryState`

The `TypedDict` are state schemas for the two LangGraph graphs. Each node receives the full state and returns only the fields it updates.

In [ ]:
from src.models.state import BuilderState, QueryState
from src.models.schemas import Chunk, Triplet, Entity, MappingProposal
import typing

# Retrieve fields defined in BuilderState
builder_fields = list(BuilderState.__annotations__.keys())
query_fields = list(QueryState.__annotations__.keys())

print(f"BuilderState — {len(builder_fields)} fields:")
for f in builder_fields:
    print(f"  · {f}: {BuilderState.__annotations__[f]}")

print(f"\nQueryState — {len(query_fields)} fields:")
for f in query_fields:
    print(f"  · {f}: {QueryState.__annotations__[f]}")

# Simulation: a node receives state and updates only its fields
def mock_extract_node(state: BuilderState) -> dict:
    """Simulates the Extract_Triplets node — updates only 'triplets'."""
    chunks = state.get("chunks", [])
    # ... extraction logic ...
    return {"triplets": []}  # returns only updated fields

initial_state: BuilderState = {
    "source_doc": "business_glossary.pdf",
    "chunks": [],
    "reflection_attempts": 0,
    "healing_attempts": 0,
    "hitl_flag": False,
    "failed_mappings": [],
    "ingestion_errors": [],
}
update = mock_extract_node(initial_state)
print(f"\nNode returns only: {list(update.keys())} (LangGraph merges into global state)")
print("✅ BuilderState and QueryState validated")

## 7. Prompt Templates — Catalog PT-01 → PT-11

All prompts are module constants in `src/prompts/templates.py`. Nodes import them directly — no inline prompts in functions.

In [ ]:
from src.prompts import templates

# List of all available templates
template_constants = [name for name in dir(templates) if name.isupper() and not name.startswith("_")]
print(f"Available templates ({len(template_constants)}):")
for name in template_constants:
    val = getattr(templates, name)
    preview = val.replace("\n", " ")[:80]
    print(f"  {name}: '{preview}...'" if len(val) > 80 else f"  {name}: '{val}'")

print()
# Show complete EXTRACTION_SYSTEM
print("=" * 60)
print("PT-01: EXTRACTION_SYSTEM (first 10 lines)")
print("=" * 60)
for line in templates.EXTRACTION_SYSTEM.split("\n")[:10]:
    print(f"  {line}")

In [ ]:
# Show how EXTRACTION_USER is formatted with real text
sample_text = "A Customer places one or more SalesOrders. Each SalesOrder contains one or more OrderLineItems."

formatted_extraction_user = templates.EXTRACTION_USER.format(chunk_text=sample_text)
print("PT-01: EXTRACTION_USER (formatted):")
print("-" * 60)
print(formatted_extraction_user)
print("-" * 60)

# Verify placeholder was replaced
assert "{chunk_text}" not in formatted_extraction_user
assert sample_text in formatted_extraction_user
print("\n✅ Placeholder correctly replaced")

## 8. Few-Shot Examples — `load_cypher_examples()` and `load_mapping_examples()`

In [ ]:
from src.prompts.few_shot import (
    load_cypher_examples,
    load_mapping_examples,
    format_cypher_examples,
    format_mapping_examples,
)

# Load Cypher examples
cypher_examples = load_cypher_examples(n=3)
print(f"Loaded Cypher examples: {len(cypher_examples)}")
for ex in cypher_examples:
    print(f"  · [{ex.concept_name}] {ex.description[:60]}")

# Load Mapping examples
mapping_examples = load_mapping_examples(n=2)
print(f"\nLoaded Mapping examples: {len(mapping_examples)}")
for ex in mapping_examples:
    print(f"  · [{ex.concept_name}] DDL snippet: {ex.ddl_snippet[:50]}...")

In [ ]:
# Format few-shot examples as text for the prompt
cypher_block = format_cypher_examples(cypher_examples[:2])
print("Few-shot Cypher (formatted for CYPHER_USER):")
print("-" * 60)
print(cypher_block[:600] + "..." if len(cypher_block) > 600 else cypher_block)
print("-" * 60)

mapping_block = format_mapping_examples(mapping_examples[:1])
print("\nFew-shot Mapping (formatted for MAPPING_USER):")
print("-" * 60)
print(mapping_block[:400] + "..." if len(mapping_block) > 400 else mapping_block)
print("-" * 60)

print("\n✅ Few-shot examples loaded and formatted correctly")

## Summary — Part 2 Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                   EP-02/06/11/15 — Data Models & Prompts            │
├──────────────────────┬──────────────────────┬───────────────────────┤
│    schemas.py        │     state.py         │  templates.py         │
│                      │                      │  + few_shot.py        │
│  Document, Chunk     │  BuilderState        │  PT-01 Extraction     │
│  Triplet             │  QueryState          │  PT-02 ER Judge       │
│  Entity, Cluster     │  (TypedDict)         │  PT-03 Mapping        │
│  TableSchema         │                      │  PT-04/05 Actor-Critic│
│  EnrichedTable       │  total=False →       │  PT-06/07 Cypher      │
│  MappingProposal     │  optional fields     │  PT-08/09 Answer      │
│  CriticDecision      │  for LangGraph       │  PT-10 Grader         │
│  CypherStatement     │  reducers            │  PT-11 Enrichment     │
│  GraderDecision      │                      │                       │
│  EvaluationReport    │                      │  few_shot_cypher.json │
│                      │                      │  few_shot_mapping.json│
└──────────────────────┴──────────────────────┴───────────────────────┘
```

**Architectural principles:**
- **Single file** for all Pydantic models → always unambiguous imports
- **TypedDict with `total=False`** → all fields optional (LangGraph node flexibility)
- **Prompts as module constants** → no inline prompts in functions
- **JSON seed files** for few-shot → versionable with Git, modifiable without touching code